In [ ]:
#packages and modules

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os

from dotenv import load_dotenv

env = load_dotenv('/mnt/d/linux/project/vars/.env')

In [ ]:
#variables

pgconn = os.getenv('POSTGRE_CONN')
pgdriver = os.getenv('PG_DRIVER')
pguser = os.getenv('POSTGRE_USER')
pgpwd = os.getenv('POSTGRE_PWD')
pgjar = os.getenv('POSTGRE_JAR')


In [ ]:
#creating session

spark = (( SparkSession
            .builder
                .appName('customers')
                .config('spark.jar', f'{pgjar}')
                .config('spark.driver.extraClassPath', f'{pgjar}')
                .config('spark.executor.extraClassPath', f'{pgjar}')
            .getOrCreate()
        ))

In [ ]:
#load customer data from source table

tmp_customerframe = (( spark
                        .read
                        .format('jdbc')
                        .option('url', pgconn)
                        .option('driver', pgdriver)
                        .option('user', pguser)
                        .option('password', pgpwd)
                        .option('query', 
                                """ 
                                
                                    select

                                        customerid,
                                        customercategoryid,
                                        buyinggroupid,
                                        billtocustomerid,
                                        split_part(customername, '(', 1)as firstname,
                                        replace(split_part(customername, '(', 2), ')', '') as lastnameOrBranch,
                                        replace(split_part(customername, ',', 2),')', '') as place,
                                        accountopeneddate,
                                        creditlimit,
                                        standarddiscountpercentage,
                                        paymentdays,
                                        primarycontactpersonid,
                                        alternatecontactpersonid,
                                        deliverymethodid,
                                        deliverycityid,
                                        postalcityid,
                                        deliverypostalcode,
                                        deliveryaddressline1,
                                        deliveryaddressline2,
                                        postaladdressline1,
                                        postaladdressline2,
                                        isstatementsent,
                                        isoncredithold,
                                        _airbyte_ab_id,
                                        _airbyte_customers_hashid,
                                        _airbyte_emitted_at,
                                        _airbyte_normalized_at
                                        
                                    from sales.customers
                                
                                """)
                        
                    )).load().withColumnRenamed('_airbyte_ab_id', 'customer_elt_hashid')

In [ ]:
#load customercategory data from source table
tmp_custmercategory = (( spark
                            .read
                            .format('jdbc')
                            .option('url', pgconn)
                            .option('driver', pgdriver)
                            .option('user', pguser)
                            .option('password', pgpwd)
                            .option('query', 
                                    
                                    """ 
                                        select

                                            customercategoryid,
                                            customercategoryname,
                                            _airbyte_ab_id,
                                            _airbyte_customercategories_hashid
                                        
                                        from sales.customercategories
                                        
                                    """)
                        )).load().withColumnRenamed('_airbyte_ab_id', 'custcategory_elt_hashid')

In [ ]:
tmp_buyinggroups = (( spark
                        .read
                        .format('jdbc')
                        .option('url', pgconn)
                        .option('driver', pgdriver)
                        .option('user', pguser)
                        .option('password', pgpwd)
                        .option('query', 
                                
                                """
                                    select
                                    
                                        buyinggroupid,
                                        buyinggroupname,
                                        _airbyte_ab_id,
                                        _airbyte_buyinggroups_hashid
                                        
                                    from sales.buyinggroups
                                        
                                """)
                        )).load().withColumnRenamed('_airbyte_ab_id', 'custgroup_elt_hashid')

In [ ]:
#joining dataframes

joined_frames = (( tmp_customerframe
                    .join(tmp_custmercategory, 'customercategoryid', 'left')
                    .join(tmp_buyinggroups, 'buyinggroupid', 'left')
                ))

In [ ]:
rename_df = ((
                joined_frames
                    .select('*', 
                            f.regexp_replace(
                                f.regexp_replace('lastnameOrBranch', f.substring_index('lastnameOrBranch', ',', -1), ''), ',', '')
                                .alias('lastnameOrBranch_n')
                            )
))

In [62]:
((
    rename_df
        .select('*', 
                f.coalesce('creditlimit', f.lit(0.00)).alias('creditlimit_n'),
                f.coalesce('buyinggroupid', f.lit(0)).alias('buyergroup_id'),
                f.coalesce('alternatecontactpersonid', f.lit(0)).alias('alternatecontact_id')
                
            )

))

DataFrame[buyinggroupid: bigint, customercategoryid: bigint, customerid: bigint, billtocustomerid: bigint, firstname: string, lastnameorbranch: string, place: string, accountopeneddate: string, creditlimit: double, standarddiscountpercentage: double, paymentdays: bigint, primarycontactpersonid: bigint, alternatecontactpersonid: bigint, deliverymethodid: bigint, deliverycityid: bigint, postalcityid: bigint, deliverypostalcode: string, deliveryaddressline1: string, deliveryaddressline2: string, postaladdressline1: string, postaladdressline2: string, isstatementsent: boolean, isoncredithold: boolean, customer_elt_hashid: string, _airbyte_customers_hashid: string, _airbyte_emitted_at: timestamp, _airbyte_normalized_at: timestamp, customercategoryname: string, custcategory_elt_hashid: string, _airbyte_customercategories_hashid: string, buyinggroupname: string, custgroup_elt_hashid: string, _airbyte_buyinggroups_hashid: string, lastnameOrBranch_n: string, creditlimit_n: double, buyergroup_id

In [61]:
renamedrop_df = ((rename_df
                     
                  
                  ))

+-------------+------------------+----------+----------------+--------------+------------------+-----+-----------------+-----------+--------------------------+-----------+----------------------+------------------------+----------------+--------------+------------+------------------+--------------------+--------------------+------------------+------------------+---------------+--------------+--------------------+-------------------------+--------------------+----------------------+--------------------+-----------------------+----------------------------------+---------------+--------------------+----------------------------+------------------+
|buyinggroupid|customercategoryid|customerid|billtocustomerid|     firstname|  lastnameorbranch|place|accountopeneddate|creditlimit|standarddiscountpercentage|paymentdays|primarycontactpersonid|alternatecontactpersonid|deliverymethodid|deliverycityid|postalcityid|deliverypostalcode|deliveryaddressline1|deliveryaddressline2|postaladdressline1|posta

In [ ]:
((
rename_df.select('*', 
                     
                     f.when((rename_df['firstname'] == "Tailspin Toys") & (rename_df['lastnameorbranch'] == "Head Office"), "Head")
                     
                     .otherwise(rename_df['firstname']).alias('pp')
                )
)).sample(withReplacement= True, fraction=0.1, seed= 230).show()

# teste.sample(0.8, 0.2).show()